# Keep sub-samples of bigger Paloma trees, add Paloma/human sequences

Author: Alexander Maksiaev

Purpose: Re-create subsampled Paloma trees, this time including Paloma and human sequences

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [9]:
# Directory paths

home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
og_fastas = home + "Alignments/" 
keep_headers = home + "Trees/subsampled_keep/"

os.chdir(keep_headers)



In [16]:
segments = ["PB2", "PB1", "PA", "HA_H1", "HA_H3", "NP", "NA_N1", "NA_N2", "MP", "NS"]
kept_segs = {}

for dirpath, dirs, files in os.walk(keep_headers):
    for file in files:
        file_name = os.path.join(dirpath, file)
        txt_name = file_name.split("/")[-1]
        for segment in segments:
            if segment in txt_name and "keep" in txt_name:
                kept_segs[segment] = file_name
                print(file_name)
    break 

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/Trees/subsampled_keep/keep_HA_H1.txt
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/Trees/subsampled_keep/keep_HA_H3.txt
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/Trees/subsampled_keep/keep_MP.txt
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/Trees/subsampled_keep/keep_NA_N1.txt
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/Trees/subsampled_keep/keep_NA_N2.txt
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/Trees/subsampled_keep/keep_NP.txt
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/Trees/subsampled_keep/keep_NS.txt
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/Trees/subsampled_keep/keep_PA.txt
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/Trees/subsampled_keep/keep_PB1.txt
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Other/Paloma/Trees/subsa

In [17]:


for dirpath, dirs, files in os.walk(og_fastas):
    for file in files:
        file_name = os.path.join(dirpath, file)
        fasta_name = file_name.split("/")[-1]
        for segment in segments:
            if segment in file_name and "trimmed_deduplicated" in file_name and segment in kept_segs.keys():
                ids_to_keep = []
                df = df_from_fasta(file_name)
                
                with open(kept_segs[segment]) as kept_segs_txt:
                    for line in kept_segs_txt.readlines():
                        id = line.split("|")[0]
                        ids_to_keep.append(id)
                        print(id)
                    kept_segs_txt.close()
                print(segment)
                # print(len(ids_to_keep))
                humans = list(df[df["full_header"].str.contains("human")]["full_header"].apply(lambda x: x.split("|")[0][1:])) # Get IDs, minus indicator ">"
                # print(len(humans))
                paloma = list(df[df["full_header"].str.contains("Iberian|White", regex=True)]["full_header"].apply(lambda x: x.split("|")[0][1:]))
                # print(len(paloma))
                ids_to_keep = ids_to_keep + humans + paloma
                print(len(ids_to_keep))
                df["id"] = df["full_header"].apply(lambda x: x.split("|")[0][1:]) 
                final_df = df[df["id"].isin(ids_to_keep)]
                df_to_fasta(final_df, segment + "_parnas_humans_Paloma.fasta", keep_headers)
    break

EPI_ISL_142645
EPI_ISL_18802200
EPI_ISL_505333
EPI_ISL_505183
EPI_ISL_505153
EPI_ISL_505142
EPI_ISL_505176
EPI_ISL_14751179
EPI_ISL_278875
EPI_ISL_207228
EPI_ISL_14751197
EPI_ISL_505087
EPI_ISL_14755056
EPI_ISL_95875
EPI_ISL_502479
EPI_ISL_502494
EPI_ISL_129513
EPI_ISL_19880217
EPI_ISL_195126
EPI_ISL_20050057
EPI_ISL_20050042
EPI_ISL_505169
EPI_ISL_231655
EPI_ISL_17646253
EPI_ISL_18885853
EPI_ISL_17646270
EPI_ISL_129478
EPI_ISL_19751572
EPI_ISL_18221379
EPI_ISL_103101
EPI_ISL_4083736
EPI_ISL_18221328
EPI_ISL_19498421
EPI_ISL_19905446
EPI_ISL_19905433
EPI_ISL_19751550
EPI_ISL_18885013
EPI_ISL_19789962
EPI_ISL_19790021
EPI_ISL_504843
EPI_ISL_9589948
EPI_ISL_16614000
EPI_ISL_381048
EPI_ISL_14751211
EPI_ISL_9593333
EPI_ISL_105985
EPI_ISL_4081352
EPI_ISL_19644483
EPI_ISL_14755058
EPI_ISL_4083804
EPI_ISL_505238
EPI_ISL_14764493
EPI_ISL_9593326
EPI_ISL_502460
EPI_ISL_504871
EPI_ISL_502445
EPI_ISL_504573
EPI_ISL_129448
EPI_ISL_14749696
EPI_ISL_501428
EPI_ISL_502545
EPI_ISL_14751169
EPI_ISL_959

In [ ]:
# Add classifications to H1 and H3

h1_classifications = pd.read_csv("cladinator_results_ha_h1_paloma.tsv", delimiter="\t")
h3_classificatiosn = pd.read_csv("cladinator_results_ha_h3_paloma.tsv", delimiter="\t")

print(h1_classifications)

for dirpath, dirs, files in os.walk(og_fastas):
    for file in files:
        file_name = os.path.join(dirpath, file)
        fasta_name = file_name.split("/")[-1]

     #Tree #                                              Query Assignment  \
0        1.0  EPIISL502479A/swine/France/29-170091/2017H1N2F...   1B.1.2.3   
1        2.0  PQ107549A/swine/Spain/17483-1/2021H1N2Spain-Gi...     1B.1.2   
2        3.0  EPIISL14764493A/swine/Italy/58247/2020H1N1Ital...     1C.2.1   
3        4.0  PQ107588A/swine/Spain/05165-2/2022H1N2Spain-vi...     1C.2.4   
4        5.0  EPIISL129466A/swine/England/17788/2000H1N1Unit...       1C.1   
..       ...                                                ...        ...   
140    141.0  EPIISL504640A/swine/Homberg/7039/2008H1N1Germa...     1C.2.4   
141    142.0  PP335507A/swine/Spain/05530-1/2020H1N2Spain-vi...     1C.2.2   
142      NaN  PQ107532A/swine/Spain/6370-9/2019H1N2Spain2019...     1C.2.2   
143      NaN  PQ107557A/swine/Spain/6370-11/2019H1N2Spain201...     1C.2.2   
144      NaN  EPIISL6781349A/swine/Spain/6370-7/2020H1N2Spai...     1C.2.2   

     Confidence              Brackets                Conclusion